In [3]:
import pandas as pd
import numpy as np

In [21]:
events_df = pd.read_csv('/Users/maciej/Desktop/MLDS_workspace/data_retailrocket/events.csv')
properites1_df = pd.read_csv('/Users/maciej/Desktop/MLDS_workspace/data_retailrocket/item_properties_part1.csv')
properites2_df = pd.read_csv('/Users/maciej/Desktop/MLDS_workspace/data_retailrocket/item_properties_part2.csv')
cat_tree_df = pd.read_table('/Users/maciej/Desktop/MLDS_workspace/data_retailrocket/category_tree.txt', sep= ',')


In [22]:
import pandas as pd

# 1. Combine and sort all properties
all_props_df = pd.concat([properites1_df, properites2_df], ignore_index=True)
all_props_df = all_props_df.sort_values('timestamp')

# 2. Extract and format Category IDs
cat_df = all_props_df[all_props_df['property'] == 'categoryid'].copy()
cat_df = cat_df[['timestamp', 'itemid', 'value']].rename(columns={'value': 'categoryid'})
cat_df['categoryid'] = pd.to_numeric(cat_df['categoryid'], errors='coerce')

# 3. Extract and format Property 790 (Price)
prop790_df = all_props_df[all_props_df['property'] == '790'].copy()
prop790_df = prop790_df[['timestamp', 'itemid', 'value']].rename(columns={'value': 'property_790'})
# Strip the 'n' and convert to float
prop790_df['property_790'] = pd.to_numeric(
    prop790_df['property_790'].str.replace('n', '', regex=False)
)

In [23]:
# Prepare the events table (must also be sorted)
table1 = events_df[['timestamp', 'visitorid', 'event', 'itemid']].copy()
table1 = table1.sort_values('timestamp')

# Point-in-time merge for categoryid — try backward first, fill gaps with forward
table1 = pd.merge_asof(
    table1, cat_df,
    on='timestamp', by='itemid',
    direction='backward'
)
# For rows still missing, fill with the next known value (forward lookup)
cat_forward = pd.merge_asof(
    table1[table1['categoryid'].isna()][['timestamp', 'itemid']],
    cat_df,
    on='timestamp', by='itemid',
    direction='forward'
).rename(columns={'categoryid': 'categoryid_fwd'})

table1 = table1.merge(cat_forward, on=['timestamp', 'itemid'], how='left')
table1['categoryid'] = table1['categoryid'].fillna(table1['categoryid_fwd'])
table1 = table1.drop(columns=['categoryid_fwd'])

# Same for property_790
table1 = pd.merge_asof(
    table1, prop790_df,
    on='timestamp', by='itemid',
    direction='backward'
)
prop_forward = pd.merge_asof(
    table1[table1['property_790'].isna()][['timestamp', 'itemid']],
    prop790_df,
    on='timestamp', by='itemid',
    direction='forward'
).rename(columns={'property_790': 'property_790_fwd'})

table1 = table1.merge(prop_forward, on=['timestamp', 'itemid'], how='left')
table1['property_790'] = table1['property_790'].fillna(table1['property_790_fwd'])
table1 = table1.drop(columns=['property_790_fwd'])

print("Table 1 Head (Point-in-Time):")
print(table1.head())

Table 1 Head (Point-in-Time):
       timestamp  visitorid      event  itemid  categoryid  property_790
0  1430622004384     693516  addtocart  297662      1130.0       14280.0
1  1430622011289     829044       view   60987       463.0      204120.0
2  1430622013048     652699       view  252860         NaN           NaN
3  1430622024154    1125936       view   33661      1628.0       32640.0
4  1430622026228     693516       view  297662      1130.0       14280.0


In [ ]:
table2 = table1.copy()

# Merge with the static category tree
table2 = table2.merge(cat_tree_df, on='categoryid', how='left')

# Fill missing parent categories with the categoryid itself
table2['parentid'] = table2['parentid'].fillna(table2['categoryid'])
table2 = table2.rename(columns={'parentid': 'parent_category'})

table2 = table2[['timestamp', 'visitorid', 'event', 'itemid', 'parent_category', 'property_790']]

print("\nTable 2 Head:")
print(table2.head())


Table 2 Head:
       timestamp  visitorid      event  itemid  parent_category  property_790
0  1430622004384     693516  addtocart  297662           1323.0       14280.0
1  1430622011289     829044       view   60987            250.0      204120.0
2  1430622013048     652699       view  252860              NaN           NaN
3  1430622024154    1125936       view   33661            605.0       32640.0
4  1430622026228     693516       view  297662           1323.0       14280.0


In [ ]:

table1_path = '/Users/maciej/Desktop/MLDS_workspace/data_retailrocket/table1_stream.csv'
table2_path = '/Users/maciej/Desktop/MLDS_workspace/data_retailrocket/table2_stream.csv'

table1.to_csv(table1_path, index=False)
table2.to_csv(table2_path, index=False)


In [26]:
table1.info()

<class 'pandas.DataFrame'>
RangeIndex: 2756659 entries, 0 to 2756658
Data columns (total 6 columns):
 #   Column        Dtype  
---  ------        -----  
 0   timestamp     int64  
 1   visitorid     int64  
 2   event         str    
 3   itemid        int64  
 4   categoryid    float64
 5   property_790  float64
dtypes: float64(2), int64(3), str(1)
memory usage: 126.2 MB
